# Notebook 2 — Reconstruction des cycles et agrégation par cycle pour Test 9 Run 1 à 3

Ce notebook part des fichiers CSV déjà extraits dans :

`C:\Users\elvas\Downloads\Stage de recherche\extracted_test9_run1`

Les objectifs sont :

1. Charger les fichiers `pwm`, `steady` et `transient` pour **Test 9 Run 1 à 3**
2. Reprendre la **même logique que dans ton notebook**
3. Reconstruire `cycle_current` à partir du signal `gateState` du bloc `pwm`
4. Associer chaque ligne du bloc `steady` au cycle correspondant grâce au temps (`timeEpoch`)
5. Agréger les données **par cycle dans chaque run**
6. Afficher :
   - le tableau agrégé du **Run 1**
   - le tableau agrégé du **Run 2**
   - le tableau agrégé du **Run 3**
   - le tableau concaténé final

## Logique générale

On adopte ici une logique de type **Unit / Cycle** :

- **Unit = run_id**
- **Cycle = cycle_current**
- **1 ligne finale = 1 cycle d'un run**

Cela signifie que le modèle verra ensuite plusieurs séquences temporelles, une par run, sans mélanger artificiellement les runs dans une même séquence.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 1. Paramètres

Tu peux modifier `DATA_DIR` si besoin, mais ici il pointe déjà vers ton dossier d'extraction.

In [2]:
DATA_DIR = r"C:\Users\elvas\Downloads\Stage de recherche\extracted_test9_run1"

RUN_IDS = [1, 2, 3,4,5,6,7]
TEST_ID = 9

CURRENT_MIN = 1.0      # filtre simple sur le courant pour éviter V/I instable
SHOW_PLOTS = True

## 2. Vérification des fichiers disponibles

In [3]:
expected_files = []
for run_id in RUN_IDS:
    prefix = f"Test_{TEST_ID}_run_{run_id}"
    expected_files.extend([
        os.path.join(DATA_DIR, f"{prefix}_pwm.csv"),
        os.path.join(DATA_DIR, f"{prefix}_steady.csv"),
        os.path.join(DATA_DIR, f"{prefix}_transient.csv"),
    ])

check_df = pd.DataFrame({
    "file_path": expected_files,
    "exists": [os.path.exists(fp) for fp in expected_files]
})

display(check_df)

,file_path,exists
0,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
1,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
2,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
3,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
4,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
5,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
6,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
7,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
8,C:\Users\elvas\Downloads\Stage de recherche\ex...,True
9,C:\Users\elvas\Downloads\Stage de recherche\ex...,True


## 3. Fonctions utilitaires

### 3.1 Chargement des trois blocs d'un run

Cette fonction charge les trois CSV :
- `pwm`
- `steady`
- `transient`

Le bloc `transient` est chargé pour vérification, mais dans cette étape la reconstruction des cycles et l'agrégation se font surtout avec `pwm` et `steady`.

In [4]:
def load_run_csvs(data_dir, test_id, run_id):
    prefix = f"Test_{test_id}_run_{run_id}"
    fp_pwm = os.path.join(data_dir, f"{prefix}_pwm.csv")
    fp_steady = os.path.join(data_dir, f"{prefix}_steady.csv")
    fp_transient = os.path.join(data_dir, f"{prefix}_transient.csv")

    df_pwm = pd.read_csv(fp_pwm)
    df_steady = pd.read_csv(fp_steady)
    df_transient = pd.read_csv(fp_transient)

    return df_pwm, df_steady, df_transient

### 3.2 Détection robuste des colonnes

Comme les noms peuvent varier légèrement d'un export à l'autre, on cherche les colonnes de manière souple.

In [5]:
def find_column(columns, candidates, required=True):
    cols_lower = {c.lower(): c for c in columns}

    for cand in candidates:
        cand_low = cand.lower()
        if cand_low in cols_lower:
            return cols_lower[cand_low]

    # recherche partielle
    for c in columns:
        c_low = c.lower()
        for cand in candidates:
            if cand.lower() in c_low:
                return c

    if required:
        raise KeyError(f"Impossible de trouver une colonne parmi : {candidates}")
    return None

### 3.3 Reconstruction du cycle à partir de `pwm`

On reprend la logique de ton notebook :

1. convertir `gateState` en variable binaire
2. détecter les **fronts montants** 0 → 1
3. faire un **cumul** des fronts montants pour obtenir `cycle_current`

Ainsi :
- chaque front montant = début d'un nouveau cycle
- le compteur de cycle augmente de 1 à chaque nouveau cycle

In [6]:
def reconstruct_cycles_from_pwm(df_pwm):
    df_pwm = df_pwm.copy()

    time_col = find_column(df_pwm.columns, ["timeEpoch", "time_epoch", "time"])
    gate_col = find_column(df_pwm.columns, ["gateState", "gate_state"])

    df_pwm[time_col] = pd.to_numeric(df_pwm[time_col], errors="coerce")
    df_pwm[gate_col] = pd.to_numeric(df_pwm[gate_col], errors="coerce")

    df_pwm = df_pwm.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

    # temps relatif en secondes
    df_pwm["t_seconds"] = (df_pwm[time_col] - df_pwm[time_col].iloc[0]) * 24 * 3600

    # binaire
    df_pwm["gateState_num"] = (df_pwm[gate_col].fillna(0) > 0).astype(int)
    df_pwm["gateState_prev"] = df_pwm["gateState_num"].shift(1).fillna(0).astype(int)

    # front montant
    df_pwm["rising_edge"] = (
        (df_pwm["gateState_num"] == 1) &
        (df_pwm["gateState_prev"] == 0)
    ).astype(int)

    # compteur de cycles
    df_pwm["cycle_current"] = df_pwm["rising_edge"].cumsum()

    # renommage uniforme pour la suite
    df_pwm = df_pwm.rename(columns={time_col: "timeEpoch"})

    return df_pwm

### 3.4 Association du cycle aux lignes `steady`

On fait ensuite un `merge_asof` entre `steady` et `pwm` sur `timeEpoch`, avec `direction="backward"`.

Cela veut dire :

> pour chaque ligne `steady`, on récupère le dernier cycle connu dans `pwm` juste avant cet instant.

C'est exactement la logique de ton notebook.

In [7]:
def attach_cycles_to_steady(df_steady, df_pwm_cycles):
    df_steady = df_steady.copy()

    time_col = find_column(df_steady.columns, ["timeEpoch", "time_epoch", "time"])
    df_steady[time_col] = pd.to_numeric(df_steady[time_col], errors="coerce")
    df_steady = df_steady.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
    df_steady = df_steady.rename(columns={time_col: "timeEpoch"})

    df_steady = pd.merge_asof(
        df_steady,
        df_pwm_cycles[["timeEpoch", "cycle_current", "t_seconds"]],
        on="timeEpoch",
        direction="backward"
    )

    df_steady["steady_t_seconds"] = (
        (df_steady["timeEpoch"] - df_steady["timeEpoch"].iloc[0]) * 24 * 3600
    )

    return df_steady

### 3.5 Calcul de `RDS_on_raw` et agrégation par cycle

Ici, on calcule :

\[
R_{DS(on), raw} = \frac{V_{DS}}{I_D}
\]

puis on agrège **par cycle** à l'intérieur de chaque run.

L'idée est la suivante :

- plusieurs lignes `steady` peuvent correspondre au même cycle
- on veut une représentation plus compacte et plus cohérente
- donc on résume chaque cycle par des statistiques robustes (ici surtout des médianes)

À la fin :
- **1 ligne = 1 cycle d'un run**

In [8]:
def build_cycle_aggregated_table(df_steady_with_cycle, run_id, current_min=1.0):
    df = df_steady_with_cycle.copy()

    # Détection souple des colonnes utiles
    current_col = find_column(df.columns, ["drainCurrent", "drain_current", "id"])
    vds_col = find_column(df.columns, ["drainSourceVoltage", "drain_source_voltage", "vds"])

    pkg_temp_col = find_column(df.columns, ["packageTemperature", "package_temperature"], required=False)
    flange_temp_col = find_column(df.columns, ["flangeTemperature", "flange_temperature"], required=False)
    supply_col = find_column(df.columns, ["supplyVoltage", "supply_voltage"], required=False)

    # conversion numérique
    for col in [current_col, vds_col, pkg_temp_col, flange_temp_col, supply_col, "cycle_current"]:
        if col is not None and col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # RDS_on brut
    df["RDS_on_raw"] = df[vds_col] / df[current_col].replace(0, np.nan)

    # filtres simples
    df = df.dropna(subset=["cycle_current", "RDS_on_raw", current_col, vds_col]).copy()
    df = df[df["cycle_current"] > 0].copy()
    df = df[df[current_col] > current_min].copy()

    agg_spec = {
        "timeEpoch": ("timeEpoch", "median"),
        "steady_t_seconds": ("steady_t_seconds", "median"),
        "t_seconds": ("t_seconds", "median"),
        "RDS_on_raw_cycle": ("RDS_on_raw", "median"),
        "RDS_on_raw_mean": ("RDS_on_raw", "mean"),
        "drainCurrent_cycle": (current_col, "median"),
        "drainSourceVoltage_cycle": (vds_col, "median"),
        "n_rows_steady": ("RDS_on_raw", "size"),
    }

    if pkg_temp_col is not None:
        agg_spec["packageTemperature_cycle"] = (pkg_temp_col, "median")
    if flange_temp_col is not None:
        agg_spec["flangeTemperature_cycle"] = (flange_temp_col, "median")
    if supply_col is not None:
        agg_spec["supplyVoltage_cycle"] = (supply_col, "median")

    df_cycle = (
        df.groupby("cycle_current", as_index=False)
          .agg(**agg_spec)
          .sort_values("cycle_current")
          .reset_index(drop=True)
    )

    df_cycle["run_id"] = run_id
    df_cycle["test_id"] = TEST_ID

    # ordre des colonnes
    preferred_first = ["test_id", "run_id", "cycle_current"]
    other_cols = [c for c in df_cycle.columns if c not in preferred_first]
    df_cycle = df_cycle[preferred_first + other_cols]

    return df_cycle, df

## 4. Traitement run par run

Dans cette partie, on applique exactement la même logique aux **Run 1, 2 et 3** :

- chargement des blocs
- reconstruction du cycle dans `pwm`
- association au `steady`
- agrégation par cycle

In [9]:
run_objects = {}
df_all_cycles_parts = []

for run_id in RUN_IDS:
    print("=" * 90)
    print(f"TRAITEMENT DU RUN {run_id}")
    print("=" * 90)

    df_pwm_raw, df_steady_raw, df_transient_raw = load_run_csvs(DATA_DIR, TEST_ID, run_id)

    print("\nDimensions brutes :")
    print(" - pwm      :", df_pwm_raw.shape)
    print(" - steady   :", df_steady_raw.shape)
    print(" - transient:", df_transient_raw.shape)

    df_pwm_cycles = reconstruct_cycles_from_pwm(df_pwm_raw)
    df_steady_with_cycle = attach_cycles_to_steady(df_steady_raw, df_pwm_cycles)
    df_cycle_agg, df_steady_filtered = build_cycle_aggregated_table(
        df_steady_with_cycle, run_id=run_id, current_min=CURRENT_MIN
    )

    run_objects[run_id] = {
        "df_pwm_raw": df_pwm_raw,
        "df_steady_raw": df_steady_raw,
        "df_transient_raw": df_transient_raw,
        "df_pwm_cycles": df_pwm_cycles,
        "df_steady_with_cycle": df_steady_with_cycle,
        "df_steady_filtered": df_steady_filtered,
        "df_cycle_agg": df_cycle_agg,
    }

    df_all_cycles_parts.append(df_cycle_agg)

    print("\nAperçu PWM avec cycle reconstruit :")
    display(df_pwm_cycles.head())

    print("\nAperçu STEADY après association au cycle :")
    display(df_steady_with_cycle.head())

    print("\nTableau agrégé par cycle pour ce run :")
    display(df_cycle_agg.head())

    print("\nNombre de cycles retenus :", len(df_cycle_agg))
    print("Cycle min / max          :", df_cycle_agg["cycle_current"].min(), "/", df_cycle_agg["cycle_current"].max())

TRAITEMENT DU RUN 1

Dimensions brutes :
 - pwm      : (1345, 11)
 - steady   : (5184, 8)
 - transient: (2728, 8)

Aperçu PWM avec cycle reconstruit :


,time,timeEpoch,nanosec_time,lowTemp,highTemp,shutdownTemp,gateVoltage,supplyVoltage,switchingFrequency,dutyCycle,gateState,t_seconds,gateState_num,gateState_prev,rising_edge,cycle_current
0,03/11/2010 16:21:56.220,734208.681901,0.221,39,40,280,10,5,1000,40,0,0.000000,0,0,0,0
1,03/11/2010 16:21:56.470,734208.681904,0.471,39,40,280,10,5,1000,40,0,0.249999,0,0,0,0
2,03/11/2010 16:21:58.154,734208.681923,0.155,39,40,280,10,5,1000,40,1,1.933997,1,0,1,1
3,03/11/2010 16:22:06.954,734208.682025,0.955,39,40,280,10,5,1000,40,0,10.733999,0,1,0,1
4,03/11/2010 16:22:07.388,734208.682030,0.389,39,40,280,10,5,1000,40,1,11.167994,1,0,1,2



Aperçu STEADY après association au cycle :


,date,nanosec_time,timeEpoch,supplyVoltage,packageTemperature,drainSourceVoltage,drainCurrent,flangeTemperature,cycle_current,t_seconds,steady_t_seconds
0,03/11/2010 16:21:56.867,0.8675,734208.681908,5.603689,29.788117,4.508691,5.179959,28.357752,0,0.249999,0.000000
1,03/11/2010 16:21:57.148,0.1485,734208.681911,5.604414,29.740844,4.531044,5.069381,32.884101,0,0.249999,0.280998
2,03/11/2010 16:21:57.550,0.5505,734208.681916,5.603912,29.867283,4.522496,5.076847,39.873114,0,0.249999,0.682998
3,03/11/2010 16:21:57.946,0.9465,734208.681921,5.604512,30.085515,4.539330,4.989771,46.263143,0,0.249999,1.078992
4,03/11/2010 16:21:58.455,0.4555,734208.681927,5.620543,30.652142,5.577962,0.087613,51.094008,1,1.933997,1.587992



Tableau agrégé par cycle pour ce run :


...


Aperçu STEADY après association au cycle :


,date,nanosec_time,timeEpoch,supplyVoltage,packageTemperature,drainSourceVoltage,drainCurrent,flangeTemperature,cycle_current,t_seconds,steady_t_seconds
0,03/13/2010 17:03:33.332,0.3325,734210.710802,5.596489,29.491043,4.771147,3.809992,26.297252,0,0.226000,0.000000
1,03/13/2010 17:03:33.613,0.6135,734210.710806,5.602558,29.551753,4.811565,3.626926,29.228058,0,0.226000,0.280998
2,03/13/2010 17:03:34.012,0.0125,734210.710810,5.599754,30.112389,4.799489,3.673302,34.798897,0,0.226000,0.680000
3,03/13/2010 17:03:34.413,0.4135,734210.710815,5.596866,31.034048,4.806876,3.659534,40.382328,0,0.226000,1.081004
4,03/13/2010 17:03:34.916,0.9165,734210.710821,5.622455,32.909097,5.580092,0.082584,45.569025,1,1.910993,1.583999



Tableau agrégé par cycle pour ce run :


,test_id,run_id,cycle_current,timeEpoch,steady_t_seconds,t_seconds,RDS_on_raw_cycle,RDS_on_raw_mean,drainCurrent_cycle,drainSourceVoltage_cycle,n_rows_steady,packageTemperature_cycle,flangeTemperature_cycle,supplyVoltage_cycle
0,9,7,1,734210.710888,7.378003,7.509997,1.318650,1.318650,3.648161,4.807554,2,35.677636,40.368603,5.596712
1,9,7,2,734210.711041,20.650999,20.778996,1.355752,1.355752,3.565400,4.828655,2,36.456748,40.917197,5.600263
2,9,7,3,734210.711268,40.185499,40.317997,1.340013,1.340013,3.598558,4.816067,2,36.239406,40.916831,5.600382
3,9,7,4,734210.711374,49.354500,46.751999,1.440272,1.417402,3.375967,4.862254,16,46.262873,76.520906,5.599908
4,9,7,5,734210.711437,54.793004,54.784998,1.516099,1.520983,3.231400,4.899124,3,64.776034,97.770654,5.600005



Nombre de cycles retenus : 4404
Cycle min / max          : 1 / 4404


## 5. Affichage détaillé des tableaux agrégés run par run

In [ ]:
for run_id in RUN_IDS:
    print("#" * 100)
    print(f"TABLEAU AGREGÉ — TEST {TEST_ID} RUN {run_id}")
    print("#" * 100)
    display(run_objects[run_id]["df_cycle_agg"])

## 6. Tableau concaténé final

On concatène maintenant les trois tables agrégées.

La logique finale est donc bien :

- `run_id` = **unit**
- `cycle_current` = **temps**
- **1 ligne = 1 cycle d'un run**

In [ ]:
df_all_cycles = pd.concat(df_all_cycles_parts, ignore_index=True)

print("Dimensions du tableau concaténé :", df_all_cycles.shape)
display(df_all_cycles.head(20))

## 7. Vérification de la structure Unit / Cycle

Ici, on vérifie que pour chaque run :
- les cycles sont bien ordonnés
- la structure est cohérente

In [ ]:
summary = (
    df_all_cycles.groupby("run_id")
    .agg(
        n_cycles=("cycle_current", "count"),
        cycle_min=("cycle_current", "min"),
        cycle_max=("cycle_current", "max"),
        rds_min=("RDS_on_raw_cycle", "min"),
        rds_max=("RDS_on_raw_cycle", "max"),
    )
    .reset_index()
)

display(summary)

## 8. Visualisation simple

On visualise l'évolution de `RDS_on_raw_cycle` en fonction du cycle pour chaque run.

In [ ]:
if SHOW_PLOTS:
    plt.figure(figsize=(10, 5))
    for run_id in RUN_IDS:
        tmp = run_objects[run_id]["df_cycle_agg"]
        plt.plot(tmp["cycle_current"], tmp["RDS_on_raw_cycle"], label=f"Run {run_id}")
    plt.xlabel("Cycle")
    plt.ylabel("RDS_on_raw_cycle")
    plt.title("Evolution de RDS_on par cycle pour Test 9 Run 1 à 3")
    plt.grid(True)
    plt.legend()
    plt.show()

## 9. Sauvegarde des tableaux agrégés

On enregistre :
- un CSV agrégé par cycle pour chaque run
- un CSV concaténé final

Cela facilitera le prochain notebook, par exemple pour :
- la normalisation
- le lissage
- la construction des séquences
- la préparation du LSTM

In [ ]:
OUT_DIR = DATA_DIR

for run_id in RUN_IDS:
    fp = os.path.join(OUT_DIR, f"Test_{TEST_ID}_run_{run_id}_cycle_agg.csv")
    run_objects[run_id]["df_cycle_agg"].to_csv(fp, index=False)
    print("Sauvegardé :", fp)

fp_all = os.path.join(OUT_DIR, f"Test_{TEST_ID}_runs_1_to_3_cycle_agg_concat.csv")
df_all_cycles.to_csv(fp_all, index=False)
print("Sauvegardé :", fp_all)

## 10. Conclusion de cette étape

À ce stade, le prétraitement demandé pour cette partie est fait :

1. les cycles ont été reconstruits à partir du bloc `pwm`
2. les lignes `steady` ont été rattachées aux cycles
3. les données ont été agrégées **par cycle et par run**
4. on a obtenu une structure de type **Unit / Cycle**

### Ce qu'il faut retenir

- **Unit = run_id**
- **Cycle = cycle_current**
- **une ligne finale = un cycle dans un run**

C'est cette structure qui servira proprement dans la suite.